In [1]:
%%writefile config.py
DATA_DIR = '/kaggle/input/bone-fracture-dataset/Bone fracture dataset/Bone fracture dataset/Dataset'
NUM_EPOCHS, BATCH_SIZE, LEARNING_RATE = 50, 32, 0.00001
MODEL_SAVE_PATH = 'bone_fracture_model_final.pth'

Writing config.py


In [2]:
%%writefile utils.py
import torch, matplotlib.pyplot as plt, seaborn as sns
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
import torchvision.transforms as transforms
from sklearn.metrics import confusion_matrix

def get_dataloaders(data_dir, batch_size):
    t = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    ds = ImageFolder(data_dir, transform=t)
    tr_len, v_len = int(0.7 * len(ds)), int(0.15 * len(ds))
    torch.manual_seed(42)
    tr_ds, v_ds, te_ds = random_split(ds, [tr_len, v_len, len(ds) - tr_len - v_len])
    print(f"Data split: Train {len(tr_ds)}, Val {len(v_ds)}, Test {len(te_ds)}")
    return DataLoader(tr_ds, batch_size, shuffle=True), DataLoader(v_ds, batch_size), DataLoader(te_ds, batch_size), ds.classes

def plot_performance_curves(h):
    plt.figure(figsize=(14, 6))
    plt.subplot(1, 2, 1); plt.plot(h['train_loss'], 'o-', label='Train'); plt.plot(h['val_loss'], 'o-', label='Val'); plt.title('Loss'); plt.legend()
    plt.subplot(1, 2, 2); plt.plot(h['train_acc'], 'o-', label='Train'); plt.plot(h['val_acc'], 'o-', label='Val'); plt.title('Accuracy'); plt.legend()
    plt.tight_layout(); plt.savefig('performance_curves.png'); plt.show(); plt.close()

def plot_confusion_matrix(model, loader, name, classes, device):
    y_p, y_t = [], []
    model.eval()
    with torch.no_grad():
        for x, y in loader:
            y_p.extend(model(x.to(device)).argmax(1).cpu().numpy()); y_t.extend(y.numpy())
    plt.figure(figsize=(8, 6))
    sns.heatmap(confusion_matrix(y_t, y_p), annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title(f'CM: {name}'); plt.savefig(f"cm_{name.replace(' ', '_')}.png"); plt.show(); plt.close()

Writing utils.py


In [3]:
%%writefile model.py
import torch, torch.nn as nn, torchvision.models as models

def create_model(device):
    model = models.efficientnet_b0(weights='DEFAULT')
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
    for param in model.parameters(): param.requires_grad = True
    return model.to(device)

Writing model.py


In [4]:
%%writefile train.py
import torch, torch.nn as nn, torch.optim as optim, time, config
from utils import get_dataloaders, plot_performance_curves, plot_confusion_matrix
from model import create_model

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    train_loader, val_loader, test_loader, classes = get_dataloaders(config.DATA_DIR, config.BATCH_SIZE)
    model = create_model(device)
    opt, crit = optim.Adam(model.parameters(), lr=config.LEARNING_RATE), nn.CrossEntropyLoss()
    hist = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    print("\nStarting model training...")
    for ep in range(config.NUM_EPOCHS):
        t0, tr_loss, tr_corr, v_loss, v_corr = time.time(), 0., 0, 0., 0
        
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            out = model(x)
            loss = crit(out, y)
            loss.backward(); opt.step()
            tr_loss += loss.item() * x.size(0); tr_corr += (out.argmax(1) == y).sum().item()
        
        model.eval()
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                out = model(x)
                v_loss += crit(out, y).item() * x.size(0); v_corr += (out.argmax(1) == y).sum().item()
        
        tr_sz, v_sz = len(train_loader.dataset), len(val_loader.dataset)
        hist['train_loss'].append(tr_loss/tr_sz); hist['train_acc'].append(100.*tr_corr/tr_sz)
        hist['val_loss'].append(v_loss/v_sz); hist['val_acc'].append(100.*v_corr/v_sz)
        print(f"Ep {ep+1}/{config.NUM_EPOCHS} [{time.time()-t0:.1f}s] | Tr Loss: {tr_loss/tr_sz:.4f}, Acc: {100.*tr_corr/tr_sz:.2f}% | Val Loss: {v_loss/v_sz:.4f}, Acc: {100.*v_corr/v_sz:.2f}%")
    
    torch.save(model.state_dict(), config.MODEL_SAVE_PATH)
    print(f"\nModel saved to {config.MODEL_SAVE_PATH}")
    
    plot_performance_curves(hist)
    for ld, nm in zip([train_loader, val_loader, test_loader], ["Train", "Val", "Test"]): 
        plot_confusion_matrix(model, ld, nm, classes, device)

if __name__ == '__main__': main()

Writing train.py


In [5]:
!python train.py


Using device: cuda
Data split: Train 1488, Val 319, Test 320
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|███████████████████████████████████████| 20.5M/20.5M [00:00<00:00, 205MB/s]

Starting model training...
Ep 1/50 [30.0s] | Tr Loss: 0.6642, Acc: 59.88% | Val Loss: 0.6050, Acc: 84.64%
Ep 2/50 [22.5s] | Tr Loss: 0.5387, Acc: 85.08% | Val Loss: 0.4615, Acc: 92.79%
Ep 3/50 [22.4s] | Tr Loss: 0.4379, Acc: 93.01% | Val Loss: 0.3818, Acc: 94.04%
Ep 4/50 [22.7s] | Tr Loss: 0.3523, Acc: 94.09% | Val Loss: 0.3131, Acc: 94.67%
Ep 5/50 [22.7s] | Tr Loss: 0.2853, Acc: 94.49% | Val Loss: 0.2588, Acc: 95.30%
Ep 6/50 [23.0s] | Tr Loss: 0.2289, Acc: 94.49% | Val Loss: 0.2132, Acc: 95.61%
Ep 7/50 [23.0s] | Tr Loss: 0.1896, Acc: 95.50% | Val Loss: 0.1858, Acc: 95.92%
Ep 8/50 [23.4s] | Tr Loss: 0.1634, Acc: 96.10% | Val Loss: 0.1548, Acc: 95.92%
Ep 9/50 [23.6s] | Tr Loss: 0.13